In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.optim as optim
from skorch import NeuralNet
from skorch.helper import SliceDataset
from sklearn.model_selection import GridSearchCV, PredefinedSplit
from torch.utils.data import ConcatDataset

import data_utils as du
import read_data
from models import HamCNN, count_trainable_parameters

# -----------------------------------------------------------------------------
# Experiment setup
# -----------------------------------------------------------------------------
RANDOM_SEED = 0
TEST_START_YEAR = 1981
TEST_LENGTH_YEARS = 30

INPUT_LENGTH = 6
LEADING_TIME = 18
INPUT_VARIABLES = ["sst"]
EXPECTED_FIELD_SHAPE = (49, 144)

TRAIN_DATASET_NAME = "NOAA_SODA_ORAS5"
TEST_DATASET_NAME = "NOAA_SODA_ORAS5"

# -----------------------------------------------------------------------------
# Spatial and temporal subset
# -----------------------------------------------------------------------------
LON_START = 0
LON_END = 357.5
LAT_START = -60
LAT_END = 60

TIME_START = "1871"
TIME_END = "2025"

# -----------------------------------------------------------------------------
# Data paths
# -----------------------------------------------------------------------------
DATA_ROOT = Path("E:/OneDrive - University of Leeds/A-Research/Study_timeseies/Data")

OBS_SST_PATH = DATA_ROOT / "Obs/SST/SST_NOAA_1871-2025.nc"
OBS_ENSO_PATH = DATA_ROOT / "Obs/ENSO/nino34_NOAA_1871-2025.nc"
OBS_OHC_PATH = DATA_ROOT / "Obs/OHC/ohc300_SODA_ORAS5_1871-2025.nc"

CMIP6_SST_DIR = DATA_ROOT / "CMIP6/SST"
CMIP6_ENSO_DIR = DATA_ROOT / "CMIP6/ENSO"
CMIP6_MEMBERS = [
    "CESM2_hist_r1",
    "ACCESS-CM2_hist_r1",
    "MIROC6_hist_r1",
]

# -----------------------------------------------------------------------------
# Parameter search
# -----------------------------------------------------------------------------
BASE_MAX_EPOCHS = 40
BASE_BATCH_SIZE = 500
BASE_LR = 0.0007
BASE_M_1 = 10
BASE_N_NUM = 50

PARAM_GRID = {
    "lr": [0.0001, 0.0005, 0.001, 0.0015, 0.002],
    "max_epochs": [15, 20, 25, 30, 35, 40],
    "optimizer": [optim.Adam, optim.AdamW],
    "batch_size": [128, 256, 512, 1024],
    "module__M_1": [16, 32, 64, 128, 256],
    "module__N_Num": [16, 32, 64, 128, 256],
}

# -----------------------------------------------------------------------------
# Outputs
# -----------------------------------------------------------------------------
RESULTS_CSV = Path("params.csv")
SORTED_RESULTS_CSV = Path("params_sorted.csv")


In [ ]:
du.set_random_seed(RANDOM_SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


In [ ]:
input_channels = INPUT_LENGTH * len(INPUT_VARIABLES)
print("Input variables:", INPUT_VARIABLES)
print("Input channels:", input_channels)
print("Leading time:", LEADING_TIME)


In [ ]:
obs_data = read_data.read_observation_data(
    obs_sst_path=OBS_SST_PATH,
    obs_enso_path=OBS_ENSO_PATH,
    obs_ohc_path=OBS_OHC_PATH,
    lat_range=(LAT_START, LAT_END),
    lon_range=(LON_START, LON_END),
    time_range=(TIME_START, TIME_END),
    include_sst="sst" in INPUT_VARIABLES,
    include_ohc="ohc" in INPUT_VARIABLES,
)
print("Loaded observation data:", list(obs_data.keys()))
for name, value in obs_data.items():
    print(name, value.shape)


In [ ]:
time_splits = du.build_time_splits(
    test_start_year=TEST_START_YEAR,
    input_length=INPUT_LENGTH,
    leading_time=LEADING_TIME,
    test_length_years=TEST_LENGTH_YEARS,
)
print("Train periods:", time_splits["train_periods"])
print("Test period:", time_splits["test_period"])
print("Target time range:", time_splits["target_time_range"])


In [ ]:
train_datasets = []
for train_start, train_end in time_splits["train_periods"]:
    train_cut = du.cut_train_data(
        obs_data,
        train_start,
        train_end,
        input_variables=INPUT_VARIABLES,
        dataset_name=TRAIN_DATASET_NAME,
    )
    train_datasets.append(
        du.make_dataset(
            train_cut,
            input_variables=INPUT_VARIABLES,
            input_length=INPUT_LENGTH,
            leading_time=LEADING_TIME,
        )
    )

train_dataset = ConcatDataset(train_datasets)

test_start, test_end = time_splits["test_period"]
test_cut = du.cut_test_data(
    obs_data,
    test_start,
    test_end,
    input_variables=INPUT_VARIABLES,
    dataset_name=TEST_DATASET_NAME,
)
test_dataset = du.make_dataset(
    test_cut,
    input_variables=INPUT_VARIABLES,
    input_length=INPUT_LENGTH,
    leading_time=LEADING_TIME,
)

search_dataset = ConcatDataset([train_dataset, test_dataset])
test_fold = np.concatenate([
    np.full(len(train_dataset), -1, dtype=int),
    np.zeros(len(test_dataset), dtype=int),
])
predefined_split = PredefinedSplit(test_fold)

sst_search = SliceDataset(search_dataset, idx=0)
enso_search = SliceDataset(search_dataset, idx=1)

sample_x, sample_y = train_dataset[0]
expected_x_shape = (input_channels, *EXPECTED_FIELD_SHAPE)
print("Train samples:", len(train_dataset))
print("Test samples:", len(test_dataset))
print("Search samples:", len(search_dataset))
print("PredefinedSplit test fold counts:", dict(zip(*np.unique(test_fold, return_counts=True))))
print("X shape:", tuple(sample_x.shape))
print("y shape:", tuple(sample_y.shape))
assert tuple(sample_x.shape) == expected_x_shape, (
    f"Expected X shape {expected_x_shape}, got {tuple(sample_x.shape)}. "
    "If lat/lon range changes, update HamCNN's linear input size manually."
)
assert tuple(sample_y.shape) == (LEADING_TIME,)


In [ ]:
model_preview = HamCNN(
    input_channels=input_channels,
    leading_time=LEADING_TIME,
    M_1=BASE_M_1,
    N_Num=BASE_N_NUM,
)
print("Trainable parameters:", count_trainable_parameters(model_preview))

net = NeuralNet(
    module=HamCNN,
    module__input_channels=input_channels,
    module__leading_time=LEADING_TIME,
    module__M_1=BASE_M_1,
    module__N_Num=BASE_N_NUM,
    criterion=torch.nn.MSELoss,
    optimizer=torch.optim.Adam,
    max_epochs=BASE_MAX_EPOCHS,
    batch_size=BASE_BATCH_SIZE,
    lr=BASE_LR,
    iterator_train__shuffle=True,
    train_split=None,
    device=device,
)


In [ ]:
grid = GridSearchCV(
    estimator=net,
    param_grid=PARAM_GRID,
    cv=predefined_split,
    scoring=du.negative_mse_first_6_months,
    verbose=2,
    n_jobs=1,
    refit=False,
)

grid_result = grid.fit(sst_search, enso_search)
print("Best score:", grid_result.best_score_)
print("Best parameters:", grid_result.best_params_)


In [ ]:
results = pd.DataFrame(grid_result.cv_results_)
results_sorted = results.sort_values(by="rank_test_score", ascending=True)

results.to_csv(RESULTS_CSV, index=False, encoding="utf-8")
results_sorted.to_csv(SORTED_RESULTS_CSV, index=False, encoding="utf-8")

print("Saved:", RESULTS_CSV)
print("Saved:", SORTED_RESULTS_CSV)
results_sorted[["rank_test_score", "mean_test_score", "params"]].head(10)
